# 06 -- Daily CRSP Stock Data Collection

## Purpose
Downloads daily stock data from CRSP for every PERMNO in the master universe list (the ~150-200 stocks that appeared in the top 100 S&P 500 by market cap at any point between 2004 and 2024). The full daily history is downloaded for each PERMNO across the entire date range, even if the stock was only in the top 100 for some years. Filtering to the specific top-100 set per year happens downstream at analysis time using `universe_annual.parquet`.

## Source
WRDS CRSP daily stock file v2 (`crsp_a_stock.dsf_v2`) via the `wrds` Python library, authenticated with username `henrylavender`.

## Input
`Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet` -- the master PERMNO list built in notebook 05.

## Collection Method
PERMNOs are queried in batches of 80 to avoid WRDS timeouts on the ~110M-row table. Each batch covers the full 2004-01-01 to 2024-12-31 date range. Results are filtered to:
- `sharetype = 'NS'` (normal shares)
- `securitytype = 'EQTY'` (equity)

## Variables Collected
- **Identifiers:** `permno`, `dlycaldt` (renamed to `date`), `ticker`, `primaryexch`
- **Prices:** `dlyprc`, `dlyopen`, `dlyhigh`, `dlylow`, `dlyclose`, `dlybid`, `dlyask`
- **Returns:** `dlyret` (total return), `dlyretx` (return excluding dividends), `dlyreti` (return including distributions)
- **Volume/Trading:** `dlyvol` (volume), `dlynumtrd` (number of trades), `dlyprcvol` (price x volume), `dlymmcnt` (market maker count)
- **Size:** `dlycap` (market capitalisation), `shrout` (shares outstanding)
- **Adjustment:** `dlyfacprc` (cumulative price adjustment factor)
- **Dividends:** `dlyorddivamt` (ordinary dividend amount), `dlynonorddivamt` (non-ordinary dividend amount)

A `year` column is added for parquet partitioning.

## Output
`Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/` -- partitioned parquet by year (subfolders `year=2004/`, `year=2005/`, etc.). Reading the top-level directory with `pd.read_parquet` returns the full DataFrame; individual years can be loaded efficiently.

In [ ]:
# %% [markdown]
# # Stage 2: Collect CRSP Daily Stock Data for the Top-100 Universe
#
# This notebook downloads daily stock data from `crsp_a_stock.dsf_v2` for every
# PERMNO in our master list (the ~150-200 stocks that appeared in the top 100
# at any point between 2004 and 2024).
#
# We download the full daily history for each PERMNO across the entire date range,
# even if the stock was only in the top 100 for some years. Filtering to the
# specific top-100 set per year happens at analysis time using `universe_annual.parquet`.
#
# Output: `data/firm_daily/` — partitioned parquet by year.

# %% [markdown]
# ## Setup

# %%
import wrds
import pandas as pd
import numpy as np
from pathlib import Path
import gc



conn = wrds.Connection(wrds_username='henrylavender')

# %% [markdown]
# ## Load the master PERMNO list
#
# This was built in Stage 1 — it contains all unique PERMNOs that appeared in
# any year's top 100 S&P 500 by market cap.

# %%
master = pd.read_parquet('../../Data/Data_Collection/Initial/05_Top100_SP500_Universe/universe_master.parquet')
permno_list = master['permno'].tolist()
print(f"Master list: {len(permno_list)} unique PERMNOs")

# %% [markdown]
# ## Define the columns to keep
#
# The dsf_v2 table has 50 columns, but many are metadata or flag columns that
# are useless as model features. We select only the columns we need:
# - Identifiers: permno, dlycaldt, ticker, primaryexch
# - Prices: dlyprc, dlyopen, dlyhigh, dlylow, dlyclose, dlybid, dlyask
# - Returns: dlyret, dlyretx, dlyreti
# - Volume/trading: dlyvol, dlynumtrd, dlyprcvol, dlymmcnt
# - Size: dlycap, shrout
# - Adjustment: dlyfacprc
# - Dividends: dlyorddivamt, dlynonorddivamt

# %%
COLUMNS = [
    'permno', 'dlycaldt',
    'dlyprc', 'dlycap', 'dlyret', 'dlyretx', 'dlyreti',
    'dlyvol', 'dlyopen', 'dlyhigh', 'dlylow', 'dlyclose',
    'dlybid', 'dlyask', 'dlynumtrd', 'dlyprcvol', 'dlymmcnt',
    'dlyfacprc', 'shrout', 'dlyorddivamt', 'dlynonorddivamt',
    'ticker', 'primaryexch'
]

col_str = ', '.join(COLUMNS)

# %% [markdown]
# ## Download the data
#
# The master list has ~150-200 PERMNOs. We query in batches of 80 to avoid
# excessively long SQL IN clauses, which can cause WRDS timeouts on a 110M-row table.
# Each batch covers the full 2004-2024 date range.

# %%
BATCH_SIZE = 80
START_DATE = '2004-01-01'
END_DATE = '2024-12-31'

batches = [permno_list[i:i+BATCH_SIZE] for i in range(0, len(permno_list), BATCH_SIZE)]
print(f"Downloading in {len(batches)} batches of up to {BATCH_SIZE} PERMNOs each")

frames = []

for i, batch in enumerate(batches):
    permno_str = ','.join(str(p) for p in batch)
    
    query = f"""
        SELECT {col_str}
        FROM crsp_a_stock.dsf_v2
        WHERE permno IN ({permno_str})
          AND dlycaldt BETWEEN '{START_DATE}' AND '{END_DATE}'
          AND sharetype = 'NS'
          AND securitytype = 'EQTY'
    """
    
    df_batch = conn.raw_sql(query, date_cols=['dlycaldt'])
    frames.append(df_batch)
    print(f"  Batch {i+1}/{len(batches)}: {len(df_batch):,} rows, "
          f"{df_batch['permno'].nunique()} PERMNOs")

df = pd.concat(frames, ignore_index=True)
del frames
gc.collect()

print(f"\nTotal downloaded: {len(df):,} rows, {df['permno'].nunique()} unique PERMNOs")

# %% [markdown]
# ## Rename and add columns
#
# - Rename `dlycaldt` → `date` for consistency across all datasets
# - Add `year` column for parquet partitioning

# %%
df = df.rename(columns={'dlycaldt': 'date'})
df['year'] = df['date'].dt.year

# Sort for clean parquet output
df = df.sort_values(['permno', 'date']).reset_index(drop=True)

print(f"Shape: {df.shape}")
print(f"Date range: {df['date'].min()} to {df['date'].max()}")

# %% [markdown]
# ## Save as partitioned parquet
#
# Partitioning by year creates subfolders `year=2004/`, `year=2005/`, etc.
# When read back with `pd.read_parquet('data/firm_daily')`, pandas treats it
# as one DataFrame but can load individual years efficiently.

# %%
# Remove existing directory if re-running
import shutil
if Path('../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily').exists():
    shutil.rmtree('../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily')

df.to_parquet('../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily', partition_cols=['year'], engine='pyarrow', index=False)

# Verify the folder structure
year_folders = sorted(Path('../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily').glob('year=*'))
print(f"Saved to ../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/ with {len(year_folders)} year partitions")
print(f"First: {year_folders[0].name}, Last: {year_folders[-1].name}")

# %% [markdown]
# ## Verification: Summary stats

# %% [markdown]
# ### Total rows, unique PERMNOs, date range

# %%
print(f"Total rows: {len(df):,}")
print(f"Unique PERMNOs: {df['permno'].nunique()}")
print(f"Date range: {df['date'].min().date()} to {df['date'].max().date()}")

# %% [markdown]
# ### Rows per year
#
# We expect roughly 25,000 rows per year (~100-200 stocks × ~252 trading days).
# Earlier years may have slightly fewer if some stocks hadn't yet entered the top 100.

# %%
rows_per_year = df.groupby('year').size()
print("Rows per year:")
print(rows_per_year.to_string())
print(f"\nMean: {rows_per_year.mean():,.0f}, Min: {rows_per_year.min():,}, Max: {rows_per_year.max():,}")

# %% [markdown]
# ### Null counts per column
#
# Most columns should have very few nulls for large-cap S&P 500 stocks.
# `dlybid`, `dlyask`, and `dlymmcnt` may have more nulls as they depend on
# exchange reporting. Dividend columns will be mostly null (dividends are infrequent).

# %%
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)
null_summary = pd.DataFrame({'null_count': null_counts, 'null_pct': null_pct})
null_summary = null_summary[null_summary['null_count'] > 0].sort_values('null_pct', ascending=False)
print("Columns with nulls:")
print(null_summary.to_string())

# %% [markdown]
# ### Sample data for AAPL (PERMNO 14593) in 2020
#
# Spot-check that prices, returns, and volume look sensible for a known stock.

# %%
aapl = df[(df['permno'] == 14593) & (df['year'] == 2020)].head(5)
print("AAPL (PERMNO 14593) — first 5 trading days of 2020:")
print(aapl[['permno', 'date', 'ticker', 'dlyprc', 'dlyret', 'dlyvol', 'dlycap']].to_string(index=False))

# %% [markdown]
# ### Check PERMNOs from master list that might be missing

# %%
downloaded_permnos = set(df['permno'].unique())
master_permnos = set(master['permno'].tolist())
missing = master_permnos - downloaded_permnos

if missing:
    print(f"WARNING: {len(missing)} PERMNOs from master list not found in CRSP download:")
    print(f"  {missing}")
    print("  (These may have been delisted before 2004 or may not meet the NS/EQTY filter)")
else:
    print("All master list PERMNOs found in CRSP download.")

# %% [markdown]
# ## Cleanup

# %%
conn.close()
del df
gc.collect()

print("\nStage 2 complete. File saved:")
print("  ../../Data/Data_Collection/Initial/06_Daily_CRSP_Stock_Data/firm_daily/ (partitioned parquet by year)")